In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import requests
from bs4 import BeautifulSoup
import re
import time
from datetime import datetime, timedelta
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, NoSuchElementException

In [ ]:
!brew uninstall chromedriver
!brew install --cask chromedriver@144

In [ ]:
# Target conferences
TARGET_CONFERENCES = ['ACC', 'SEC', 'B10', 'B12', 'P12', 'BE']

# Date range
START_DATE = datetime(2022, 11, 1)
END_DATE = datetime(2023, 3, 20)

def setup_driver():
    """Setup Chrome driver with options"""
    chrome_options = Options()
    chrome_options.add_argument('--headless')  # Run in background
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-blink-features=AutomationControlled')
    chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
    
    driver = webdriver.Chrome(options=chrome_options)
    return driver
def get_date_range():
    """Generate list of dates to scrape"""
    dates = []
    current = START_DATE
    while current <= END_DATE:
        dates.append(current)
        current += timedelta(days=1)
    return dates

def format_date_for_url(date):
    """Format date as YYYYMMDD for URL"""
    return date.strftime('%Y%m%d')

def scrape_day(driver, date):
    """Scrape data for a single day"""
    date_str = format_date_for_url(date)
    url = f"https://barttorvik.com/?year=2026&sort=&hteam=&t2value=&conlimit=All&state=All&begin={date_str}&end={date_str}&top=0&revquad=0&quad=5&venue=All&type=All&mingames=0"
    
    print(f"Scraping {date.strftime('%Y-%m-%d')}...")
    
    try:
        driver.get(url)
        
        # Wait for the table to load
        wait = WebDriverWait(driver, 20)
        
        # Wait for table to be present
        try:
            table = wait.until(EC.presence_of_element_located((By.TAG_NAME, "table")))
        except TimeoutException:
            print(f"  Timeout waiting for table on {date.strftime('%Y-%m-%d')}")
            return None
        
        # Give it a moment for JavaScript to finish rendering
        time.sleep(2)
        
        # Get the table HTML
        table_html = driver.find_element(By.TAG_NAME, "table").get_attribute('outerHTML')
        
        # Parse with pandas - skip the first row (D-I averages) and handle multi-level headers
        dfs = pd.read_html(table_html, header=1)  # Use row 1 as header, skip row 0
        
        if not dfs:
            print(f"  No table found for {date.strftime('%Y-%m-%d')}")
            return None
        
        df = dfs[0]  # Get first table
        
        # If there are still duplicate columns, keep only the first occurrence of each
        if df.columns.duplicated().any():
            df = df.loc[:, ~df.columns.duplicated(keep='first')]
        
        # Filter for target conferences
        conf_col = None
        for col in df.columns:
            if 'conf' in str(col).lower() or 'league' in str(col).lower():
                conf_col = col
                break
        
        if conf_col:
            # Filter by conference
            df = df[df[conf_col].isin(TARGET_CONFERENCES)]
        else:
            # If no specific conference column, filter by checking all columns
            mask = df.astype(str).apply(lambda row: any(conf in ' '.join(row.values) for conf in TARGET_CONFERENCES), axis=1)
            df = df[mask]
        
        if len(df) == 0:
            print(f"  No matching conference data found for {date.strftime('%Y-%m-%d')}")
            return None
        
        # Add scraped date column
        df['Date'] = date.strftime('%Y-%m-%d')
        
        print(f"  Found {len(df)} rows with {len(df.columns)} columns")
        return df
        
    except Exception as e:
        print(f"  Error scraping {date.strftime('%Y-%m-%d')}: {e}")
        import traceback
        traceback.print_exc()
        return None

In [ ]:
def main():
    """Main scraping function"""
    print("Starting barttorvik.com scraper (Selenium)")
    print(f"Date range: {START_DATE.strftime('%Y-%m-%d')} to {END_DATE.strftime('%Y-%m-%d')}")
    print(f"Target conferences: {', '.join(TARGET_CONFERENCES)}")
    print("-" * 60)
    
    dates = get_date_range()
    print(f"Total days to scrape: {len(dates)}")
    print("-" * 60)
    
    # Setup driver
    print("Initializing Chrome driver...")
    driver = setup_driver()
    
    all_data = []
    
    try:
        for i, date in enumerate(dates, 1):
            df = scrape_day(driver, date)
            if df is not None and len(df) > 0:
                all_data.append(df)
            
            # Be polite - add delay between requests
            if i < len(dates):
                time.sleep(2)
        
        print("-" * 60)
        
        if all_data:
            # Combine all data - this just stacks rows
            combined_df = pd.concat(all_data, ignore_index=True)
            
            print(f"\nCombined dataframe shape: {combined_df.shape}")
            print(f"Columns: {len(combined_df.columns)}")
            
            # Save to CSV
            output_file = 'barttorvik_data.csv'
            combined_df.to_csv(output_file, index=False)
            
            print(f"\nSuccess! Scraped {len(combined_df)} total rows")
            print(f"Data saved to: {output_file}")
            print(f"\nFirst few rows:")
            print(combined_df.head())
            
            # Show unique dates scraped
            print(f"\nUnique dates scraped: {combined_df['Date'].nunique()}")
        else:
            print("\nNo data was scraped successfully")
    
    finally:
        # Always close the driver
        driver.quit()
        print("\nDriver closed")

if __name__ == "__main__":
    main()

In [ ]:
torvik = pd.read_csv("barttorvik_data.csv", header = 0)
torvik.columns

torvik['Team'] = torvik['Team'].str.split('  ').str[0].str.strip()
torvik.head(n = 10)

In [ ]:
torvik.to_csv("barttorvik_data_23.csv")